# 01 — Download and preprocessing

ENTSO-E Transparency Platform → `data/dataset.parquet` (protocol: CLAUDE.md, section 5).

| Series | `entsoe-py` method | Dataset column |
|---|---|---|
| Day-ahead price, DE-LU | `query_day_ahead_prices` | `price` (EUR/MWh) |
| Day-ahead total load forecast | `query_load_forecast(process_type="A01")` | `load_fc` (GW) |
| Day-ahead wind onshore + offshore forecast | `query_wind_and_solar_forecast(process_type="A01")` | `wind_fc` (GW) |
| Day-ahead solar forecast | same call | `solar_fc` (GW) |

Steps:
1. Download month by month and cache the raw series **in UTC** under `data/raw/` (re-runs do not query the API again).
2. Aggregate to hourly by averaging the quarter-hours **in UTC**; MW → GW; `wind_fc` = onshore + offshore.
3. Missing values: report per series, linearly interpolate gaps of at most 3 h, report longer gaps.
4. Convert to Berlin local time with **exactly 24 hours per day** (DST convention below).
5. Add the calendar: `hour`, `dow`, `non_working_day`.
6. Checks, Energy-Charts spot check, save.

Negative prices and spikes are real values: nothing is filtered or clipped.

In [ ]:
import os
import time

import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

from src import config as cfg

RAW_DIR = cfg.DATA_DIR / "raw"
UTC = "UTC"

## 1. Download period

From `DOWNLOAD_START` to the last complete day. Once set in `src/config.py`, `CONTROL_END` freezes the end of the download so that both machines build the same dataset. If it is `None`, the download runs up to today (Berlin): the prices of day D are published on D−1, so today is complete once the auction of yesterday has cleared.

In [ ]:
last_day = cfg.CONTROL_END if cfg.CONTROL_END is not None else pd.Timestamp.now(tz=cfg.TZ).normalize().tz_localize(None)
START = pd.Timestamp(cfg.DOWNLOAD_START, tz=cfg.TZ)  # inclusive, Berlin midnight
END = pd.Timestamp(last_day + pd.Timedelta(days=1), tz=cfg.TZ)  # exclusive, Berlin midnight
print(f"Download: {START} -> {END} (exclusive), last day {last_day.date()}")

## 2. Download with a monthly cache

The API key is read from `.env` (`ENTSOE_API_KEY`) and never printed. Each series is requested month by month (Berlin month boundaries) and each chunk is stored as `data/raw/<series>/<start>_<end>.parquet`, with its index converted to UTC. A chunk already on disk is not downloaded again; the last, partial month gets a new file name when the end of the download moves.

`query_day_ahead_prices` includes the `end` timestamp, the forecast methods do not: every chunk is cut to `[start, end)`, then duplicated timestamps at the chunk boundaries are dropped.

In [ ]:
load_dotenv(cfg.ROOT / ".env")
client = EntsoePandasClient(api_key=os.environ["ENTSOE_API_KEY"])

QUERIES = {
    "price": lambda a, b: client.query_day_ahead_prices(cfg.ZONE, start=a, end=b).to_frame("price"),
    "load_forecast": lambda a, b: client.query_load_forecast(cfg.ZONE, start=a, end=b, process_type="A01"),
    "wind_solar_forecast": lambda a, b: client.query_wind_and_solar_forecast(
        cfg.ZONE, start=a, end=b, psr_type=None, process_type="A01"
    ),
}


def month_chunks(start, end):
    # Month boundaries at Berlin midnight; first and last chunks may be partial
    inner = pd.date_range(start, end, freq="MS", inclusive="neither")
    bounds = [start, *inner, end]
    return list(zip(bounds[:-1], bounds[1:]))


def fetch(name, start, end):
    # Raw series in UTC, at the native resolution (hourly or 15 min), cached month by month
    frames = []
    for a, b in month_chunks(start, end):
        path = RAW_DIR / name / f"{a:%Y%m%d}_{b:%Y%m%d}.parquet"
        if path.exists():
            chunk = pd.read_parquet(path)
        else:
            try:
                chunk = QUERIES[name](a, b)
            except NoMatchingDataError:
                print(f"{name}: no data for {a.date()} -> {b.date()}")
                continue
            chunk.index = chunk.index.tz_convert(UTC)
            chunk = chunk[(chunk.index >= a) & (chunk.index < b)]  # drop the extra `end` point
            path.parent.mkdir(parents=True, exist_ok=True)
            chunk.to_parquet(path)
        frames.append(chunk)
    raw = pd.concat(frames).sort_index()
    return raw[~raw.index.duplicated(keep="first")]


raw = {name: fetch(name, START, END) for name in QUERIES}
for name, df in raw.items():
    print(f"{name:20s} {len(df):7d} rows  {list(df.columns)}  {df.index.min()} -> {df.index.max()}")

### Native resolution

The DE-LU day-ahead market moved from hourly to 15-minute periods on 1 October 2025. The table counts the time steps between consecutive raw timestamps, per series and per year.

In [ ]:
steps = pd.concat(
    {
        name: pd.Series(df.index[1:] - df.index[:-1], index=df.index[1:]).groupby(df.index[1:].year).value_counts()
        for name, df in raw.items()
    },
    names=["series", "year", "step"],
)
steps.unstack("step", fill_value=0)

## 3. Hourly aggregation in UTC

Each UTC hour is the mean of the values whose timestamp falls in it: the 4 quarter-hours at 15-minute resolution, the single value at hourly resolution. Averaging in UTC avoids any ambiguity on the autumn DST day, where a local hour occurs twice. Then MW → GW and `wind_fc` = onshore + offshore (missing if either is missing, so that a gap is not hidden as a zero).

In [ ]:
utc_index = pd.date_range(START.tz_convert(UTC), END.tz_convert(UTC), freq="h", inclusive="left")


def to_hourly(df):
    return df.groupby(df.index.floor("h")).mean().reindex(utc_index)


price_h = to_hourly(raw["price"])
load_h = to_hourly(raw["load_forecast"])
ws_h = to_hourly(raw["wind_solar_forecast"])

hourly = pd.DataFrame(
    {
        "price": price_h["price"],
        "load_fc": load_h["Forecasted Load"] / 1000,
        "wind_fc": (ws_h["Wind Onshore"] + ws_h["Wind Offshore"]) / 1000,
        "solar_fc": ws_h["Solar"] / 1000,
    },
    index=utc_index,
)
hourly.index.name = "datetime_utc"
hourly.describe().T

Incomplete hours at 15-minute resolution (fewer than 4 quarter-hours present) are still averaged over the quarters available; an hour with no value at all becomes missing. The forecasts are at 15-minute resolution over the whole period, the prices since 2025-10-01.

In [ ]:
QUARTER_HOURLY_SINCE = {"price": pd.Timestamp("2025-10-01", tz=cfg.TZ), "load_forecast": START, "wind_solar_forecast": START}


def incomplete_hours(df, since):
    df = df[df.index >= since]
    counts = df.groupby(df.index.floor("h")).count()
    return (counts.lt(4) & counts.gt(0)).sum()


pd.concat([incomplete_hours(df, QUARTER_HOURLY_SINCE[name]) for name, df in raw.items()]).rename(
    "hours with 1-3 quarter-hours"
)

## 4. Missing values

Counted on the regular UTC hourly grid, per series. Gaps of at most `MAX_INTERP_GAP_HOURS` = 3 h are linearly interpolated between their neighbours; longer gaps are listed and handled explicitly below.

In [ ]:
def gaps(s):
    # One row per run of consecutive missing hours
    isna = s.isna()
    run = (isna != isna.shift()).cumsum()[isna]
    grouped = s.index[isna].to_series().groupby(run.values)
    return pd.DataFrame({"start": grouped.first(), "hours": grouped.size()}).reset_index(drop=True)


def missing_report(df):
    rows = {}
    for col in df.columns:
        g = gaps(df[col])
        rows[col] = {
            "missing hours": int(df[col].isna().sum()),
            "gaps": len(g),
            f"gaps <= {cfg.MAX_INTERP_GAP_HOURS} h": int((g["hours"] <= cfg.MAX_INTERP_GAP_HOURS).sum()),
            f"gaps > {cfg.MAX_INTERP_GAP_HOURS} h": int((g["hours"] > cfg.MAX_INTERP_GAP_HOURS).sum()),
            "longest gap (h)": int(g["hours"].max()) if len(g) else 0,
        }
    return pd.DataFrame(rows).T


missing_report(hourly)

In [ ]:
# Every gap, with the action taken
all_gaps = pd.concat({col: gaps(hourly[col]) for col in hourly.columns}, names=["series"]).droplevel(1)
all_gaps.assign(
    start_berlin=all_gaps["start"].dt.tz_convert(cfg.TZ),
    action=np.where(all_gaps["hours"] <= cfg.MAX_INTERP_GAP_HOURS, "interpolated", "kept missing, see below"),
)

In [ ]:
def fill_short_gaps(s, max_gap):
    # Linear interpolation only inside gaps of at most `max_gap` hours, between their two neighbours
    isna = s.isna()
    run = (isna != isna.shift()).cumsum()
    run_len = isna.groupby(run).transform("sum")
    fillable = isna & (run_len <= max_gap)
    return s.where(~fillable, s.interpolate(method="linear", limit_area="inside"))


filled = hourly.apply(fill_short_gaps, max_gap=cfg.MAX_INTERP_GAP_HOURS)
print("Interpolated hours per series:")
print((hourly.isna() & filled.notna()).sum().to_string())
missing_report(filled)

**Gaps found** (download up to 2026-11-08): the prices are complete. Every forecast gap is a single hour, 00:00 local time on an autumn DST day (2023, 2024 and, for wind and solar, 2025). The hour is missing at the source: ENTSO-E returns no value for it even when only that day is requested, so the monthly chunking is not the cause. These hours are linearly interpolated (gaps ≤ 3 h); none is in the main test or control period.

**Decision on gaps longer than 3 h**: there are none. If a later download finds a longer gap, it is listed in the table above and left missing: inside the main test or control period it stops the notebook (assertion in section 7) and must be decided explicitly.

## 5. Berlin local time, exactly 24 hours per day

The working table uses the convention of `epftoolbox`: every day has 24 hourly values, indexed by a **tz-naive timestamp in Berlin local time** (a tz-aware index cannot hold 24 rows on a DST day).

- **Spring DST day** (last Sunday of March, 23 h): the local hour 02:00 does not exist; its value is linearly interpolated between 01:00 and 03:00.
- **Autumn DST day** (last Sunday of October, 25 h): the local hour 02:00 occurs twice (CEST then CET); the two values are averaged.

The main test period (2026-04-01 → 2026-09-30) contains no DST day; the control period contains 2026-10-25.

In [ ]:
def to_local_24h(df_utc):
    local = df_utc.tz_convert(cfg.TZ)
    naive = local.index.tz_localize(None).floor("h")  # local wall-clock hour
    out = local.groupby(naive).mean()  # autumn: averages the two occurrences of 02:00
    grid = pd.date_range(naive.min().normalize(), naive.max().normalize() + pd.Timedelta(hours=23), freq="h")
    out = out.reindex(grid)
    holes = ~grid.isin(naive)  # spring: 02:00 does not exist
    out.loc[holes] = out.interpolate(method="linear", limit_area="inside").loc[holes]
    out.index.name = "datetime"
    return out, grid[holes]


dataset, spring_holes = to_local_24h(filled)
print("Interpolated spring hours:", [str(t) for t in spring_holes])

dst_days = pd.Series(filled.index.tz_convert(cfg.TZ).tz_localize(None).normalize()).value_counts().loc[lambda c: c != 24]
print("Days that do not have 24 local hours before the conversion:")
print(dst_days.sort_index().to_string())

per_day = dataset.groupby(dataset.index.normalize()).size()
assert (per_day == 24).all() and dataset.index.is_monotonic_increasing and dataset.index.is_unique

In [ ]:
# Example: autumn 2025 DST day around 02:00 local time
day = pd.Timestamp("2025-10-26")
utc_rows = filled.loc[filled.index.tz_convert(cfg.TZ).tz_localize(None).normalize() == day].copy()
utc_rows["local"] = utc_rows.index.tz_convert(cfg.TZ)
display(utc_rows.iloc[1:5])
display(dataset.loc[day + pd.Timedelta(hours=1) : day + pd.Timedelta(hours=3)])

## 6. Calendar

- `hour`: hour of day 0–23, Berlin local time;
- `dow`: day of week, 0 = Monday … 6 = Sunday;
- `non_working_day`: 1 on Saturday, Sunday or a German **national** public holiday (`holidays.Germany()` without a subdivision, so regional holidays such as Corpus Christi are not included), 0 otherwise.

In [ ]:
years = range(dataset.index.year.min(), dataset.index.year.max() + 1)
de_holidays = holidays.Germany(years=years)
holiday_dates = pd.DatetimeIndex(sorted(de_holidays))

days = dataset.index.normalize()
dataset["hour"] = dataset.index.hour.astype(int)
dataset["dow"] = dataset.index.day_of_week.astype(int)
dataset["non_working_day"] = ((dataset["dow"] >= 5) | days.isin(holiday_dates)).astype(int)
dataset = dataset[[cfg.TARGET, *cfg.COVARIATES, *cfg.CALENDAR]]

pd.Series({d.date(): de_holidays[d] for d in holiday_dates}, name="national holiday").to_frame()

## 7. Checks

In [ ]:
periods = {
    "pre-test": (cfg.PRETEST_START, cfg.PRETEST_END),
    "main test": (cfg.TEST_START, cfg.TEST_END),
    "control": (cfg.CONTROL_START, last_day),
}


def period(df, start, end):
    return df.loc[start : end + pd.Timedelta(hours=23)]


summary = pd.DataFrame(
    {
        name: {
            "days": len(period(dataset, a, b)) // 24,
            "rows": len(period(dataset, a, b)),
            **{f"NaN {c}": int(period(dataset, a, b)[c].isna().sum()) for c in dataset.columns},
            "negative price hours": int((period(dataset, a, b)["price"] < 0).sum()),
            "min price": period(dataset, a, b)["price"].min(),
            "max price": period(dataset, a, b)["price"].max(),
        }
        for name, (a, b) in periods.items()
    }
).T
summary

In [ ]:
assert list(dataset.columns) == ["price", "load_fc", "wind_fc", "solar_fc", "hour", "dow", "non_working_day"]
assert (dataset.groupby(dataset.index.normalize()).size() == 24).all()
assert len(period(dataset, cfg.TEST_START, cfg.TEST_END)) == 183 * 24
for name in ["main test", "control"]:
    assert not period(dataset, *periods[name]).isna().any().any(), f"NaN in the {name} period"

# Last complete day: last day with 24 non-missing values for every series
complete = dataset[[cfg.TARGET, *cfg.COVARIATES]].notna().all(axis=1).groupby(dataset.index.normalize()).all()
last_complete_day = complete[complete].index.max()
print("Last complete day:", last_complete_day.date())
assert last_complete_day == last_day, "set CONTROL_END in src/config.py to the last complete day"

### Energy-Charts spot check

Prices from the Energy-Charts API (Fraunhofer ISE, `https://api.energy-charts.info/price`, bidding zone `DE-LU`), aggregated to hourly in UTC exactly as above, compared to the ENTSO-E hourly prices on three days:
- one day at hourly resolution (before 2025-10-01);
- one day at 15-minute resolution (after 2025-10-01);
- the day with the most negative-price hours of the dataset.

In [ ]:
def energy_charts_hourly(day, retries=5):
    # Hourly UTC prices of one Berlin day from Energy-Charts, cached under data/raw/energy_charts/
    path = RAW_DIR / "energy_charts" / f"{day:%Y%m%d}.parquet"
    if path.exists():
        return pd.read_parquet(path)["price"]
    a = pd.Timestamp(day, tz=cfg.TZ)
    b = a + pd.Timedelta(days=1)
    for attempt in range(retries):
        r = requests.get(
            "https://api.energy-charts.info/price",
            params={"bzn": "DE-LU", "start": a.isoformat(), "end": b.isoformat()},
            timeout=60,
        )
        if r.status_code != 429:  # rate limit: wait and retry
            break
        time.sleep(30 * (attempt + 1))
    r.raise_for_status()
    j = r.json()
    s = pd.Series(j["price"], index=pd.to_datetime(j["unix_seconds"], unit="s", utc=True), dtype=float)
    s = s[(s.index >= a) & (s.index < b)]
    s = s.groupby(s.index.floor("h")).mean().rename("price")
    path.parent.mkdir(parents=True, exist_ok=True)
    s.to_frame().to_parquet(path)
    return s


neg_hours = (dataset["price"] < 0).groupby(dataset.index.normalize()).sum()
check_days = [pd.Timestamp("2024-06-12"), pd.Timestamp("2026-02-11"), neg_hours.idxmax()]

rows = []
for day in check_days:
    ec = energy_charts_hourly(day)
    ent = hourly["price"].reindex(ec.index)
    diff = (ec - ent).abs()
    rows.append({"day": day.date(), "hours compared": diff.notna().sum(), "negative hours": int((ent < 0).sum()),
                 "max |diff| (EUR/MWh)": diff.max(), "mean |diff| (EUR/MWh)": diff.mean()})
spot_check = pd.DataFrame(rows).set_index("day")
spot_check

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
day = check_days[-1]
ec = energy_charts_hourly(day)
ax.plot(ec.index.tz_convert(cfg.TZ), ec.values, "o-", label="Energy-Charts", alpha=0.7)
ax.plot(ec.index.tz_convert(cfg.TZ), hourly["price"].reindex(ec.index).values, "x--", label="ENTSO-E")
ax.axhline(0, color="grey", lw=0.8)
ax.set(title=f"Hourly DE-LU day-ahead price, {day.date()}", ylabel="EUR/MWh")
ax.legend()
plt.tight_layout()

## 8. Save

In [ ]:
dataset.to_parquet(cfg.DATASET_PATH)
check = pd.read_parquet(cfg.DATASET_PATH)
pd.testing.assert_frame_equal(check, dataset, check_freq=False)
print(cfg.DATASET_PATH.relative_to(cfg.ROOT), check.shape, check.index.min(), "->", check.index.max())
check.dtypes